## 4. Descriptive Analysis & Visualizations

### 4.1 Summary Statistics

In [ ]:
print("Numerical Summary Statistics:")
display(df[numerical_cols].describe())

print("Categorical Summary (e.g., Cancellation Rate):")
print(df['is_canceled'].value_counts(normalize=True))

# Descriptive statistics: Cancelled vs Non-Cancelled comparison
print("\nDescriptive Statistics — Cancelled vs Non-Cancelled Bookings:")
comparison_cols = ['lead_time', 'adr', 'total_nights', 'total_of_special_requests', 'booking_changes', 'days_in_waiting_list']
cancelled_compare = df.groupby('is_canceled')[comparison_cols].agg(['mean', 'median', 'std']).T
cancelled_compare.columns = ['Not Cancelled (0)', 'Cancelled (1)']
display(cancelled_compare)

### 4.2 Visualizations

In [ ]:
sns.set_palette('Set2')
sns.set_style('whitegrid')

# 1. Bar charts of cancellation rates by market segment and hotel type
plt.figure(figsize=(10, 6))
sns.barplot(x='market_segment', y='is_canceled', hue='hotel', data=df, errorbar=None)
plt.title("Cancellation Rates by Market Segment and Hotel Type")
plt.xticks(rotation=45)
plt.ylabel("Cancellation Rate")
plt.show()

# 2. Line charts of monthly ADR trends split by hotel type
monthly_stats = df.groupby(['arrival_date_month', 'hotel'])['adr'].mean().reset_index()
# Sort months logically
monthly_stats['arrival_date_month'] = pd.Categorical(monthly_stats['arrival_date_month'], categories=list(month_map.keys()), ordered=True)
monthly_stats = monthly_stats.sort_values('arrival_date_month')

plt.figure(figsize=(10, 6))
sns.lineplot(x='arrival_date_month', y='adr', hue='hotel', data=monthly_stats, marker='o', linewidth=2)
plt.title("Average Daily Rate (ADR) by Month and Hotel Type")
plt.xticks(rotation=45)
plt.ylabel("Avg ADR (\u20ac)")
plt.legend(title='Hotel Type')
plt.show()

# 3. Box plots of lead time by cancellation status
plt.figure(figsize=(8, 6))
sns.boxplot(x='is_canceled', y='lead_time', data=df)
plt.title("Lead Time by Cancellation Status")
plt.xticks([0, 1], ['Not Canceled', 'Canceled'])
plt.ylabel("Lead Time (Days)")
plt.show()

# 4. Pie charts of deposit type distribution
deposit_counts = df['deposit_type'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(deposit_counts, labels=deposit_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title("Distribution of Deposit Types")
plt.show()

# 5. Count plots of guest country of origin (Top 10)
top_countries = df['country'].value_counts().nlargest(10).index
plt.figure(figsize=(10, 6))
sns.countplot(y='country', data=df[df['country'].isin(top_countries)], order=top_countries, palette='viridis')
plt.title("Top 10 Guest Countries of Origin")
plt.xlabel("Number of Bookings")
plt.ylabel("Country Code")
plt.show()

# 6. Heatmaps of variable correlations
plt.figure(figsize=(12, 10))
corr_cols = numerical_cols + ['is_canceled', 'room_mismatch']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Key Variables")
plt.show()

# 7. Distributions of key variables split by cancellation status
dist_cols = ['lead_time', 'adr', 'total_nights', 'total_of_special_requests']
dist_labels = ['Lead Time (Days)', 'ADR (\u20ac)', 'Total Nights', 'Special Requests']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(dist_cols, dist_labels)):
    sns.histplot(data=df, x=col, hue='is_canceled', bins=40, kde=True,
                 ax=axes[i], palette={0: 'steelblue', 1: 'tomato'}, alpha=0.6)
    axes[i].set_title(f'Distribution of {label} by Cancellation Status')
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Count')
    legend = axes[i].get_legend()
    if legend:
        legend.set_title('Cancelled')
        for t, l in zip(legend.texts, ['No', 'Yes']):
            t.set_text(l)

plt.suptitle("Key Variable Distributions: Cancelled vs Not Cancelled", fontsize=14)
plt.tight_layout()
plt.show()